# AEGIS Kaggle Runner

This notebook runs the local `inference/` package on Kaggle with the single-model multi-agent pipeline.


## 1. Install inference dependencies

This notebook intentionally avoids installing `unsloth` in the inference runtime.

`gemma4` requires a recent `transformers` release, so this notebook installs a Gemma4-compatible version instead of the older 4.x line.


In [ ]:
import os
import subprocess
import sys

os.environ["TORCH_FLEX_ATTENTION"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers>=5.5.3",
        "sentence-transformers>=2.6.0",
        "bitsandbytes>=0.43.0",
        "peft>=0.10.0",
        "accelerate>=0.27.0",
        "pillow",
        "requests",
    ],
    check=True,
)

print("Dependencies installed.")


## 2. Load Hugging Face token from Kaggle Secrets


In [ ]:
try:
    from kaggle_secrets import UserSecretsClient

    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = hf_token
    print(f"HF_TOKEN loaded from Kaggle Secrets (length={len(hf_token)})")
except Exception as exc:
    print(f"HF_TOKEN was not loaded automatically: {exc}")
    print("If the models are gated, add HF_TOKEN in Kaggle Secrets and re-run this cell.")


## 3. Locate the project and import the pipeline

This searches common Kaggle locations so the notebook still works whether the code is uploaded as a dataset or copied into `/kaggle/working`.


In [ ]:
from pathlib import Path
import sys


def find_project_root() -> Path:
    direct_candidates = [
        Path.cwd(),
        Path("/kaggle/working"),
    ]

    for candidate in direct_candidates:
        if (candidate / "inference" / "agentic_pipeline.py").exists():
            return candidate

    input_root = Path("/kaggle/input")
    if input_root.exists():
        for match in input_root.rglob("agentic_pipeline.py"):
            if match.parent.name == "inference":
                return match.parent.parent

    raise FileNotFoundError(
        "Could not find the project root containing inference/agentic_pipeline.py"
    )


PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")

from inference.agentic_pipeline import create_agentic_pipeline


## 4. Configure and initialize the pipeline

Use the GRPO adapter by default. If you want a more conservative baseline, try the SFT adapter instead.


In [ ]:
MODEL_NAME = "unsloth/gemma-4-E2B-it"
MODEL_ADAPTER = "SamSankar/aegis-gemma4-e2b-grpo-v2"
# MODEL_ADAPTER = "SamSankar/aegis-gemma4-e2b-sft-v2"
MAX_DEBATE_ROUNDS = 3
ENGINE_TYPE = "hf"

pipeline = create_agentic_pipeline(
    model_name=MODEL_NAME,
    model_adapter=MODEL_ADAPTER,
    max_debate_rounds=MAX_DEBATE_ROUNDS,
    engine_type=ENGINE_TYPE,
)


## 5. Choose an image

Set either `IMAGE_PATH` for a Kaggle file or `IMAGE_URL` for a remote image.


In [ ]:
IMAGE_PATH = ""
IMAGE_URL = "https://images.unsplash.com/photo-1514888286974-6c03e2ca1dba?w=640"


In [ ]:
import io

import requests
from IPython.display import display
from PIL import Image

if IMAGE_PATH:
    img = Image.open(IMAGE_PATH).convert("RGB")
elif IMAGE_URL:
    response = requests.get(
        IMAGE_URL,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30,
    )
    response.raise_for_status()
    img = Image.open(io.BytesIO(response.content)).convert("RGB")
else:
    raise ValueError("Set IMAGE_PATH or IMAGE_URL before running this cell.")

print(f"Loaded image: size={img.size}, mode={img.mode}")
display(img)


## 6. Run inference


In [ ]:
import gc

import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

result = pipeline.analyze(img)

peak_vram_gb = None
if torch.cuda.is_available():
    peak_vram_gb = torch.cuda.max_memory_allocated() / 1e9

summary = {
    "verdict": result["verdict"],
    "display_label": result.get("display_label", result["verdict"]),
    "confidence": result.get("confidence"),
    "label_confidence": result.get("label_confidence"),
    "manipulation_risk": result.get("manipulation_risk", result.get("aegis_score")),
    "aegis_score": result.get("aegis_score"),
    "debate_score": result.get("debate_score"),
    "nli_score": result.get("nli_score"),
    "nli_max_contradiction": result.get("nli_max_contradiction"),
    "nli_mean_entailment": result.get("nli_mean_entailment"),
    "nli_mean_neutral": result.get("nli_mean_neutral"),
    "groundedness_penalty": result.get("groundedness_penalty"),
    "evidence_region": result.get("evidence_region"),
    "peak_vram_gb": None if peak_vram_gb is None else round(peak_vram_gb, 2),
}

summary


## 7. Inspect the reasoning trace and per-claim NLI scores


In [ ]:
print("=" * 60)
print("DISPLAY LABEL")
print("=" * 60)
print(result.get("display_label", result["verdict"]))
print(f"Label Confidence: {result.get('label_confidence', result['confidence']):.3f}")
print(f"Manipulation Risk: {result.get('manipulation_risk', result.get('aegis_score', 0.0)):.3f}")
print(f"AEGIS Score: {result.get('aegis_score', 0.0):.3f}")
print(f"NLI Score: {result.get('nli_score', 0.0):.3f}")
print()

print("=" * 60)
print("CHAIN-OF-THOUGHT TRACE")
print("=" * 60)
print(result.get("cot_trace", ""))

print()
print("=" * 60)
print("PER-CLAIM NLI SCORES")
print("=" * 60)

for claim in result.get("per_claim_scores", []):
    scores = claim["scores"]
    print(f"Round {claim['round']}")
    print(f"  Label: {claim['verdict_label']}")
    print(f"  Contradiction: {scores['contradiction']:.3f}")
    print(f"  Entailment: {scores['entailment']:.3f}")
    print(f"  Neutral: {scores['neutral']:.3f}")
    print(f"  Support Score: {claim.get('support_score', 0.0):.3f}")
    print(f"  Challenge Score: {claim.get('challenge_score', 0.0):.3f}")
    print(f"  Hypothesis: {claim['hypothesis'][:240]}")
    print()


## 8. Cleanup


In [ ]:
pipeline.unload()
print("Pipeline unloaded.")
